In [ ]:
# 0-0-0-1-run-plink2.py

In [ ]:
import os, subprocess, sys
import numpy as np
import pandas as pd

from mpire import WorkerPool

def run_plink2snp(chr_name, from_bp, end_bp, gene_name):
    cmd = "bash /share/home/zhoujb/project/riceDataComb/processData/dataFromScience/0-0-0-1-run-plink2.sh {} {} {} {}".format(chr_name, from_bp, end_bp, gene_name)
    subprocess.run(cmd, shell=True)
    return 

if __name__ == "__main__":
    input_list = []

    if len(sys.argv) < 3:
        print("Usage: python script.py <input_file_path> <num threads>")
        sys.exit(1)

    in_file_path = sys.argv[1]
    info_file = pd.read_table(in_file_path)
    for index in info_file.index:
        Chr = int(info_file.loc[index, "chr"])
        Start = int(info_file.loc[index, "start"])
        End = int(info_file.loc[index, "end"])
        gene_name = info_file.loc[index, "Name"]
        input_list.append((Chr, Start, End, gene_name))

    with WorkerPool(n_jobs=int(sys.argv[2])) as pool:
        pool.map(run_plink2snp, input_list, progress_bar=True)

In [ ]:
# 0-0-0-1-run-plink2.sh

In [ ]:
#!/bin/bash

# Configure
ulimit -s unlimited

chr_name=$1
from_bp=$2
end_bp=$3
gene_name=$4

source /share/home/zhoujb/anaconda3/etc/profile.d/conda.sh

plink2 \
  --vcf /share/home/zhoujb/project/riceDataComb/processData/dataFromScience/newid_science.vcf \
  --chr ${chr_name} \
  --from-bp ${from_bp} \
  --to-bp ${end_bp} \
  --snps-only \
  --threads 32 \
  --export vcf \
  --out /share/home/zhoujb/project/riceDataComb/processData/dataFromScience/150GeneHap/${gene_name}

echo "DONE!"
